# TradaBoostR2

This notebook runs TradaBoostR2 as implemented in https://adapt-python.github.io/adapt/generated/adapt.utils.make_regression_da.html. 

Make sure to install adapt package (preferrably with Python 3.9), along with Tensorflow == 2.15. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

from adapt.instance_based import TrAdaBoostR2, TwoStageTrAdaBoostR2
from sklearn.metrics import mean_squared_error, mean_absolute_error

import itertools

from sklearn.preprocessing import StandardScaler

In [2]:
seed_list = [1]
splitting_variable_list = ['CRIM'] #, 'PTRATIO', 'LSTAT']


In [3]:
data = pd.read_csv('datasets/boston-housing.csv')
data.columns

data = data.dropna()
for col in data.select_dtypes(include=['object']).columns:
    data[col] = data[col].astype('category').cat.codes
data

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
5,0.02985,0.0,2.18,0.0,0.458,6.430,58.7,6.0622,3,222,18.7,394.12,5.21,28.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,0.17783,0.0,9.69,0.0,0.585,5.569,73.5,2.3999,6,391,19.2,395.77,15.10,17.5
500,0.22438,0.0,9.69,0.0,0.585,6.027,79.7,2.4982,6,391,19.2,396.90,14.33,16.8
502,0.04527,0.0,11.93,0.0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0.0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9


In [4]:
#print correlations with target
target_column = 'MEDV'
correlations = data.corr()[target_column].drop(target_column)
print(correlations)
data.columns

CRIM      -0.397230
ZN         0.406822
INDUS     -0.510829
CHAS       0.173701
NOX       -0.459054
RM         0.723951
AGE       -0.407470
DIS        0.279547
RAD       -0.416638
TAX       -0.508864
PTRATIO   -0.543809
B          0.347256
LSTAT     -0.743450
Name: MEDV, dtype: float64


Index(['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX',
       'PTRATIO', 'B', 'LSTAT', 'MEDV'],
      dtype='object')

In [5]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Reshape
from tensorflow.keras.optimizers import Adam

def get_model():
    model = Sequential()
    model.add(Dense(12, activation='relu', input_shape=(12,)))
    model.add(Dense(12, activation='relu'))
    model.add(Dense(12, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer=Adam(1e-3), loss='mean_squared_error')
    return model

In [ ]:
#ablation study for TradaBoostR2, Gaussian errors, with gaussian source domain errors
ablation_transfer_tradaboost_normal_normal = pd.DataFrame(columns = ['seed', 'splitting_variable', 'method',
                                   'n_estimators', 'lr', 'epochs', 'val_rmse', 'val_mae', 'rmse', 'mae'])

n_estimators_list = [5,20,35]
lr_list = [0.05, 0.1, 0.15]
epochs_list = [10, 30, 50]


# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    n_estimators_list,
    lr_list,
    epochs_list
))

for seed in seed_list:
    for splitting_variable in splitting_variable_list:
    
        #order based on this variable
        data_ = data.sort_values(by=splitting_variable)
        #remove this variable now
        data_ = data_.drop(columns = splitting_variable)
        #divide into source and target data
        data_source = data_[0:int(len(data_) / 3)]
        data_target = data_[int(2*len(data_) / 3):]

        predictor_columns = [col for col in data_.columns if col != target_column]

        #Split target into train, val, test

        data_target_train, data_target_temp = train_test_split(data_target, test_size = 0.8, random_state=seed)
        data_target_val, data_target_test = train_test_split(data_target_temp, test_size = 0.5, random_state=seed)
        print(len(data_target_train), len(data_target_val), len(data_target_test))


        # --- Source domain ---
        X_source_train = np.array(data_source[predictor_columns], dtype=float)
        y_source_train = np.array(data_source[target_column])

        # Clean NaNs/Infs before scaling
        X_source_train = np.nan_to_num(X_source_train, nan=0.0, posinf=0.0, neginf=0.0)

        scaler_source = StandardScaler()
        X_source_train = scaler_source.fit_transform(X_source_train)


        # --- Target domain ---
        X_target_train = np.array(data_target_train[predictor_columns], dtype=float)
        y_target_train = np.array(data_target_train[target_column])

        X_target_val = np.array(data_target_val[predictor_columns], dtype=float)
        y_target_val = np.array(data_target_val[target_column])

        X_target_test = np.array(data_target_test[predictor_columns], dtype=float)
        y_target_test = np.array(data_target_test[target_column])

        # Clean NaNs/Infs before scaling
        X_target_train = np.nan_to_num(X_target_train, nan=0.0, posinf=0.0, neginf=0.0)
        X_target_val = np.nan_to_num(X_target_val, nan=0.0, posinf=0.0, neginf=0.0)
        X_target_test = np.nan_to_num(X_target_test, nan=0.0, posinf=0.0, neginf=0.0)

        # Fit scaler only on training data, then transform all target splits
        scaler_target = StandardScaler()
        X_target_train = scaler_target.fit_transform(X_target_train)
        X_target_val = scaler_target.transform(X_target_val)
        X_target_test = scaler_target.transform(X_target_test)

        for config in param_grid:
            n_estimators, lr, epochs = config


            method = f'TradaBoostR2'
            model = TrAdaBoostR2(get_model(),
                            n_estimators=n_estimators, lr=lr)

            model.fit(X_source_train, y_source_train, X_target_train, y_target_train, epochs = epochs, batch_size=16, verbose=0)
            preds = model.predict(X_target_test)
            val_preds = model.predict(X_target_val)
            val_rmse = np.sqrt(mean_squared_error(val_preds, y_target_val))
            val_mae = mean_absolute_error(val_preds, y_target_val)
            rmse = np.sqrt(mean_squared_error(preds, y_target_test))
            mae = mean_absolute_error(preds, y_target_test)
            ablation_transfer_tradaboost_normal_normal.loc[len(ablation_transfer_tradaboost_normal_normal)] = [seed, splitting_variable, method, n_estimators,
                                                                                                                lr, epochs, val_rmse, val_mae, rmse, mae]
            ablation_transfer_tradaboost_normal_normal.to_csv(f'results/tradaboost_ablation_housing.csv')

78 59 60


Iteration 0 - Error: 0.2084
Iteration 1 - Error: 0.2151
Iteration 2 - Error: 0.2220
Iteration 3 - Error: 0.2293
Iteration 4 - Error: 0.2303
Iteration 0 - Error: 0.2440
Iteration 1 - Error: 0.2408
Iteration 2 - Error: 0.2433
Iteration 3 - Error: 0.2571
Iteration 4 - Error: 0.2595
Iteration 0 - Error: 0.1658
Iteration 1 - Error: 0.1743
Iteration 2 - Error: 0.1768
Iteration 3 - Error: 0.1878
Iteration 4 - Error: 0.1914
Iteration 0 - Error: 0.2162
Iteration 1 - Error: 0.2271
Iteration 2 - Error: 0.2313
Iteration 3 - Error: 0.2393
Iteration 4 - Error: 0.2452
Iteration 0 - Error: 0.1819
Iteration 1 - Error: 0.1941
Iteration 2 - Error: 0.2004
Iteration 3 - Error: 0.2296
Iteration 4 - Error: 0.2181
Iteration 0 - Error: 0.2037
Iteration 1 - Error: 0.2027
Iteration 2 - Error: 0.2142
Iteration 3 - Error: 0.2335
Iteration 4 - Error: 0.2457
Iteration 0 - Error: 0.2047
Iteration 1 - Error: 0.2167
Iteration 2 - Error: 0.2308
Iteration 3 - Error: 0.2444
Iteration 4 - Error: 0.2627
Iteration